# Buyer classification EDA

The label is 1 when a customer acquires any new product next month, otherwise 0. Here, “buyer” means a new acquisition; it does not tell us whether an offer caused it.

Explore profiles on October–March training pairs. Use April to choose the threshold and May to evaluate it. All inputs come from the month before the label.

[General EDA](eda.ipynb) · [Modeling and MAP@7](modeling.ipynb) · [Setup](../README.md)

Use a fresh kernel and run one notebook at a time. This notebook now uses 5,000 customers to reduce memory use. The saved 50,000-customer benchmark results are separate.

In [ ]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

from datetime import date
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, ConfusionMatrixDisplay
from recommender_santander.utils import get_project_root
from recommender_santander.columns import PRODUCTS
from recommender_santander.data import load_sample, monthly_pairs
from recommender_santander.modeling import feature_frame, fit_buyer_classifier, choose_buyer_threshold, classifier_metrics

PROJECT_DIR = get_project_root()
source = PROJECT_DIR / "data/raw/train/train_ver2.csv.zip"
if not source.exists():
    source = source.with_suffix("")

In [ ]:
sample = load_sample(source, customers=5_000, seed=42)
pairs = monthly_pairs(sample).with_columns(
    (pl.sum_horizontal([f"new__{product}" for product in PRODUCTS]) > 0).alias("is_buyer"),
    pl.sum_horizontal(PRODUCTS).alias("products_owned"),
)
training = pairs.filter(pl.col("target_date").is_between(date(2015, 10, 1), date(2016, 3, 31)))
validation = pairs.filter(pl.col("target_date") == date(2016, 4, 28))
test = pairs.filter(pl.col("target_date") == date(2016, 5, 28))
display(pd.DataFrame([
    {"split": name, "pairs": len(frame), "buyers": frame["is_buyer"].sum(),
     "buyer_pct": round(frame["is_buyer"].mean() * 100, 2)}
    for name, frame in [("Training", training), ("April", validation), ("May", test)]
]))

## Class balance

Always predicting “no acquisition” can produce high accuracy without finding a single buyer.

In [ ]:
monthly = pairs.group_by("target_date").agg(
    (pl.col("is_buyer").mean() * 100).alias("buyer_pct")
).sort("target_date")
monthly.to_pandas().plot(x="target_date", y="buyer_pct", marker="o", legend=False, figsize=(9, 3))
plt.ylabel("Customer-month pairs with an acquisition (%)")
plt.tight_layout()
plt.show()
print(f"Training buyer rate: {100 * training['is_buyer'].mean():.2f}%")
print(f"Always predicting no acquisition: {100 * (1-training['is_buyer'].mean()):.2f}% accuracy, 0% buyer recall")

## Buyer and non-buyer profiles

Compare medians and missingness using training data only.

In [ ]:
profiles = training.group_by("is_buyer").agg(
    pl.len().alias("pairs"),
    pl.col("age").median().alias("median_age"),
    pl.col("tenure_months").median().alias("median_tenure"),
    pl.col("household_income").median().alias("median_income"),
    pl.col("products_owned").mean().alias("mean_products_owned"),
    (pl.col("household_income").is_null().mean() * 100).alias("income_missing_pct"),
)
display(profiles.sort("is_buyer").to_pandas())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, column in zip(axes, ["age", "tenure_months", "household_income"]):
    upper = training[column].quantile(.99)
    bins = np.linspace(0, max(upper, 1), 31)
    for buyer, label in [(False, "No acquisition"), (True, "Acquisition")]:
        values = training.filter(pl.col("is_buyer") == buyer)[column].drop_nulls().to_numpy()
        ax.hist(values[values <= upper], bins=bins, density=True, histtype="step", label=label)
    ax.set_title(column.replace("_", " "))
axes[0].legend()
plt.tight_layout()
plt.show()

The distributions are normalized separately. For these plots only, omit missing values and values above the training 99th percentile. Training keeps the full cleaned range.

## Activity, tenure, and existing products

Rates below pool several training months. Customers repeat, and differences are descriptive rather than causal.

In [ ]:
training = training.with_columns(
    pl.col("is_active_customer").replace({"0": "Inactive", "1": "Active"}).fill_null("Unknown").alias("activity"),
    pl.when(pl.col("tenure_months").is_null()).then(pl.lit("Unknown"))
    .when(pl.col("tenure_months") < 6).then(pl.lit("Under 6 months"))
    .when(pl.col("tenure_months") < 24).then(pl.lit("6–23 months"))
    .when(pl.col("tenure_months") < 60).then(pl.lit("24–59 months"))
    .otherwise(pl.lit("60+ months")).alias("tenure_group"),
    pl.when(pl.col("products_owned") >= 5).then(pl.lit("5+"))
    .otherwise(pl.col("products_owned").cast(pl.Int32).cast(pl.String)).alias("holdings_group"),
)

In [ ]:
for feature in ["activity", "tenure_group", "holdings_group"]:
    rates = training.group_by(feature).agg(
        pl.len().alias("pairs"), (pl.col("is_buyer").mean() * 100).alias("buyer_pct"),
    ).sort(feature).to_pandas()
    display(rates)
    rates.plot.bar(x=feature, y="buyer_pct", legend=False, rot=30, figsize=(7, 2.5))
    plt.axhline(training["is_buyer"].mean() * 100, linestyle="--", color="gray")
    plt.ylabel("Acquire next month (%)")
    plt.tight_layout()
    plt.show()

Active customer-months have a higher acquisition rate than inactive ones. Existing products may also help explain which customers acquire something next.

## Fit the buyer classifier

Fit on October–March, then choose the threshold that maximizes F1 on April. Do not load the API artifact here: it was refitted through May.

Precision measures buyers among selected customers. Recall measures how many buyers are reached. Coverage measures the share of customers selected.

In [ ]:
buyer_model = fit_buyer_classifier(training, seed=42)
threshold = choose_buyer_threshold(buyer_model, validation)
validation_y = validation["is_buyer"].to_numpy()
test_y = test["is_buyer"].to_numpy()
validation_scores = buyer_model.predict_proba(feature_frame(validation.to_pandas()))[:, 1]
test_scores = buyer_model.predict_proba(feature_frame(test.to_pandas()))[:, 1]
print(f"April-selected threshold: {threshold:.4f}")

## Precision–recall curves

The dashed line is the buyer prevalence. Average precision summarizes the curve across thresholds; the red point shows our April-selected threshold.

In [ ]:
for name, actual, scores in [("April", validation_y, validation_scores), ("May", test_y, test_scores)]:
    precision, recall, _ = precision_recall_curve(actual, scores)
    metrics = classifier_metrics(actual, scores, threshold)
    plt.figure(figsize=(6, 3))
    plt.plot(recall, precision, label=f"Average precision: {metrics['average_precision']:.3f}")
    plt.axhline(actual.mean(), color="gray", linestyle="--", label="Buyer prevalence")
    plt.scatter(metrics["recall"], metrics["precision"], color="red")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(name)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Threshold tradeoff

Compare cutoffs on April only. The best F1 threshold is a starting point; an offer budget or contact cost could lead to a different choice.

In [ ]:
thresholds = sorted({.05, .10, threshold, .25, .50})
tradeoffs = pd.DataFrame([
    classifier_metrics(validation_y, validation_scores, cutoff) for cutoff in thresholds
])
display(tradeoffs[["threshold", "selected_customers", "precision", "recall", "coverage", "f1"]])

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    test_y, test_scores >= threshold,
    display_labels=["No acquisition", "Acquisition"], values_format="d", cmap="Blues",
)
plt.title("May: threshold chosen on April")
plt.tight_layout()
plt.show()
may = classifier_metrics(test_y, test_scores, threshold)
display(pd.DataFrame([may]))
print(f"Selected: {may['coverage']:.2%}; buyer precision: {may['precision']:.2%}; buyer recall: {may['recall']:.2%}")

At this threshold, most buyers are still missed. Higher precision among a selected group should not be confused with better MAP@7 over all customers.

## Buyer-model feature importance

These scores come from the buyer classifier, not the product ranker. Correlated features can share importance.

In [ ]:
importance = pd.DataFrame({
    "feature": buyer_model.feature_names_, "importance": buyer_model.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance.head(10))
importance.head(10).sort_values("importance").plot.barh(x="feature", y="importance", legend=False, figsize=(8, 3))
plt.tight_layout()
plt.show()

The classifier reduces offers but misses buyers. Results cover one historical holdout month and customers with complete consecutive product labels. Scores have not been checked for calibration, and acquisition prediction does not prove an offer caused a purchase.

The [modeling notebook](modeling.ipynb) shows the three MAP@7 populations and the whole-population score after abstention.